# Predicción del Tiempo Real de Entrega de Pedidos (Lead Time Logístico)

Dataset: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

Estructura del notebook: sigue la numeración de la consigna (secciones 1 a 9, ver `tarea.md`).

| Sección | Estado |
|---|---|
| 1. Contexto del negocio | hecho |
| 2. Objetivo principal | hecho |
| 3. Variable objetivo (y) | hecho |
| 4. Construcción del dataset (X) | hecho |
| 5. Procesamiento y limpieza | pendiente |
| 6. Estrategia de validación | pendiente |
| 7. Modelos | pendiente |
| 8. Métricas | pendiente |
| 9. Entregables | pendiente |

## 1. Contexto del Negocio

Para una plataforma de comercio electrónico como Olist, estimar con precisión el tiempo de entrega es clave para gestionar la experiencia del usuario y optimizar la logística. La plataforma ya calcula una fecha de entrega estimada con reglas predefinidas. Un modelo de Machine Learning permite estimar de forma continua el tiempo real de entrega, en días, desde el momento exacto de la compra.

## 2. Objetivo Principal

Desarrollar, evaluar y comparar modelos de regresión supervisados para predecir la variable continua `y`: el tiempo total de entrega, en días, desde la fecha de compra hasta la entrega efectiva en el domicilio del cliente.

## 3. Definición de la Variable Objetivo (y)

$$y = \frac{\texttt{order\_delivered\_customer\_date} - \texttt{order\_purchase\_timestamp}}{24\ \text{horas}}$$

Se filtran las órdenes no entregadas (`order_status != 'delivered'`) y las que no tienen fecha de entrega.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")  # carpeta con los CSV de Olist

In [2]:
# tabla de pedidos, con las fechas leídas como fechas
orders = pd.read_csv(
    DATA / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
    ],
)

print("Órdenes totales:", len(orders))
print(orders["order_status"].value_counts())

Órdenes totales: 99441
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [3]:
# nos quedamos con las órdenes entregadas y con fecha real de entrega
orders = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notna())
].copy()

# y en días (con decimales): total_seconds() / segundos de un día
orders["y"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

print("Órdenes entregadas:", len(orders))
orders["y"].describe()

Órdenes entregadas: 96470


count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
25%          6.766204
50%         10.217477
75%         15.720182
max        209.628611
Name: y, dtype: float64

In [4]:
# solo inspeccionamos los valores raros de y: se tratan en la sección 5
print("y negativos:", (orders["y"] < 0).sum())
print("y > 90 días:", (orders["y"] > 90).sum())
print("y > 120 días:", (orders["y"] > 120).sum())
print("\nPercentiles de y:")
print(orders["y"].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(2))

y negativos: 0
y > 90 días: 77
y > 120 días: 43

Percentiles de y:
0.01     1.83
0.25     6.77
0.50    10.22
0.75    15.72
0.99    46.05
Name: y, dtype: float64


## 4. Construcción del Dataset de Entrenamiento (X)

Unificamos las tablas del dataset relacional para construir las variables predictoras a nivel de orden (`order_id`). Cada subsección (A a D) produce una tabla con una fila por orden, y al final se unen todas.

In [5]:
# carga del resto de las tablas
customers = pd.read_csv(DATA / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(DATA / "olist_order_payments_dataset.csv")
products = pd.read_csv(DATA / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA / "olist_geolocation_dataset.csv")

# nos quedamos solo con los ítems de las órdenes que pasaron el filtro de la sección 3
order_items = order_items[order_items["order_id"].isin(orders["order_id"])].copy()
order_payments = order_payments[order_payments["order_id"].isin(orders["order_id"])].copy()

print("Ítems:", len(order_items), "| Pagos:", len(order_payments))

Ítems: 110189 | Pagos: 100748


### A. Variables Geográficas y Logísticas

- **Distancia (km):** Haversine entre las coordenadas del vendedor y del comprador.
- **Mismo estado:** 1 si `customer_state == seller_state`, 0 en caso contrario.

Antes de agregar `geolocation` descartamos las coordenadas que caen fuera de Brasil: en la primera versión la distancia máxima daba 8 678 km, más que cualquier distancia posible dentro del país (aprox. 4 400 km).

In [6]:
# coordenadas dentro de Brasil (caja aproximada)
geo = geolocation[
    geolocation["geolocation_lat"].between(-34, 6)
    & geolocation["geolocation_lng"].between(-74, -34)
]
print("Filas de geolocation fuera de Brasil descartadas:", len(geolocation) - len(geo))

# una fila por código postal: promedio de lat y lng
geo_agg = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
    .reset_index()
)

# dos copias con nombres distintos: una para el comprador y otra para el vendedor
geo_cli = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "lat": "lat_cli", "lng": "lng_cli",
})
geo_ven = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "lat": "lat_ven", "lng": "lng_ven",
})

Filas de geolocation fuera de Brasil descartadas: 42


In [7]:
# destino: pedido + cliente + coordenadas del cliente
destino = (
    orders[["order_id", "customer_id"]]
    .merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]],
           on="customer_id", how="inner")
    .merge(geo_cli, on="customer_zip_code_prefix", how="left")
)


def haversine_km(lat1, lng1, lat2, lng2):
    """Distancia en km entre dos puntos (latitud y longitud en grados)."""
    radio_tierra = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return 2 * radio_tierra * np.arcsin(np.sqrt(a))


# ítems con datos del vendedor y del comprador
items_geo = (
    order_items[["order_id", "order_item_id", "seller_id"]]
    .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]],
           on="seller_id", how="left")
    .merge(geo_ven, on="seller_zip_code_prefix", how="left")
    .merge(destino[["order_id", "customer_state", "lat_cli", "lng_cli"]],
           on="order_id", how="inner")
)

items_geo["distancia_km"] = haversine_km(
    items_geo["lat_cli"], items_geo["lng_cli"], items_geo["lat_ven"], items_geo["lng_ven"]
)
items_geo["mismo_estado"] = (items_geo["customer_state"] == items_geo["seller_state"]).astype(int)

# una fila por orden
# mismo_estado = 1 solo si TODOS los ítems vienen del estado del comprador (min)
feat_geo = items_geo.groupby("order_id").agg(
    distancia_km=("distancia_km", "mean"),
    mismo_estado=("mismo_estado", "min"),
    customer_state=("customer_state", "first"),
    seller_state=("seller_state", "first"),
).reset_index()

print(feat_geo.shape)
feat_geo["distancia_km"].describe()

(96470, 5)


count    95993.000000
mean       600.414282
std        592.055775
min          0.000000
25%        187.747506
50%        433.839063
75%        797.530165
max       3399.190906
Name: distancia_km, dtype: float64

### B. Variables del Producto e Ítems

- **Volumen total:** (largo × alto × ancho) / 1000, en dm³.
- **Peso total:** suma de `product_weight_g` por orden.
- **Cantidad de ítems** (`count_items`).
- **Diversidad de vendedores:** vendedores distintos en el mismo pedido.

Se usa `sum(min_count=1)` para que una orden cuyos productos no tienen peso o dimensiones quede como `NaN` y no como 0.

In [8]:
# volumen de cada producto en dm3
products["volumen_dm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
) / 1000

items_prod = order_items[["order_id", "order_item_id", "product_id", "seller_id"]].merge(
    products[["product_id", "product_category_name", "product_weight_g", "volumen_dm3"]],
    on="product_id", how="left",
)

feat_prod = items_prod.groupby("order_id").agg(
    volumen_dm3=("volumen_dm3", lambda s: s.sum(min_count=1)),
    peso_g=("product_weight_g", lambda s: s.sum(min_count=1)),
    count_items=("order_item_id", "count"),
    n_vendedores=("seller_id", "nunique"),
    product_category_name=("product_category_name", "first"),  # categoría del primer ítem
).reset_index()

print(feat_prod.shape)
feat_prod.head()

(96470, 6)


,order_id,volumen_dm3,peso_g,count_items,n_vendedores,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,3.528,650.0,1,1,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,60.000,30000.0,1,1,pet_shop
2,000229ec398224ef6ca0657da4fc703e,14.157,3050.0,1,1,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,2.400,200.0,1,1,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,42.000,3750.0,1,1,ferramentas_jardim


### C. Variables Financieras

- **Precio total** de la orden: suma de `price`.
- **Valor total del flete:** suma de `freight_value`.
- **Ratio flete/precio:** `flete_total / precio_total`.
- **Cuotas de pago:** máximo de `payment_installments` por orden.

In [9]:
feat_items = order_items.groupby("order_id").agg(
    precio_total=("price", "sum"),
    flete_total=("freight_value", "sum"),
).reset_index()

feat_pagos = order_payments.groupby("order_id").agg(
    cuotas_max=("payment_installments", "max"),
).reset_index()

feat_fin = feat_items.merge(feat_pagos, on="order_id", how="left")

# replace(0, nan) evita dividir por cero si algún pedido tuviera precio 0
feat_fin["ratio_flete_precio"] = feat_fin["flete_total"] / feat_fin["precio_total"].replace(0, np.nan)

print(feat_fin.shape)
feat_fin.head()

(96470, 5)


,order_id,precio_total,flete_total,cuotas_max,ratio_flete_precio
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,2.0,0.225637
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,3.0,0.083076
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,5.0,0.089799
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,2.0,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,3.0,0.090745


### D. Variables Temporales

Extraídas de `order_purchase_timestamp`: día de la semana (0 a 6), hora (0 a 23), mes (1 a 12) y binario de fin de semana.

In [10]:
ts = orders["order_purchase_timestamp"]

feat_temp = orders[["order_id"]].copy()
feat_temp["dia_semana"] = ts.dt.dayofweek          # 0 = lunes
feat_temp["hora"] = ts.dt.hour
feat_temp["mes"] = ts.dt.month
feat_temp["es_fin_de_semana"] = (ts.dt.dayofweek >= 5).astype(int)

feat_temp.head()

,order_id,dia_semana,hora,mes,es_fin_de_semana
0,e481f51cbdc54678b7cc49136f2d6af7,0,10,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,1,20,7,0
2,47770eb9100c2d0c44946d9cf07ec65d,2,8,8,0
3,949d5b44dbf5de918fe9c16f97b45f8a,5,19,11,1
4,ad21c59c0840e6cb83a9ceb5573f8159,1,21,2,0


### Unión final
Se unen la variable objetivo y las cuatro tablas de variables. Se conservan `order_purchase_timestamp` (para la división temporal de la sección 6) y `order_estimated_delivery_date` (solo para la comparación de negocio de la sección 9; no es una variable predictora).

In [11]:
base = orders[["order_id", "order_purchase_timestamp", "order_estimated_delivery_date", "y"]]

df = (
    base
    .merge(feat_geo, on="order_id", how="inner")
    .merge(feat_prod, on="order_id", how="left")
    .merge(feat_fin, on="order_id", how="left")
    .merge(feat_temp, on="order_id", how="left")
)

print("Filas:", len(df), "| order_id únicos:", df["order_id"].nunique())
assert len(df) == df["order_id"].nunique(), "Hay órdenes duplicadas: revisar los joins"

print("\nValores faltantes por columna:")
print(df.isna().sum()[lambda s: s > 0])

Filas: 96470 | order_id únicos: 96470

Valores faltantes por columna:
distancia_km              477
volumen_dm3                16
peso_g                     16
product_category_name    1332
cuotas_max                  1
dtype: int64


In [12]:
# verificación de la distancia tras limpiar las coordenadas
print("Órdenes con distancia > 4500 km:", (df["distancia_km"] > 4500).sum())
print("Distancia máxima:", round(df["distancia_km"].max()), "km")

print(df.shape)
df.head()

Órdenes con distancia > 4500 km: 0
Distancia máxima: 3399 km
(96470, 21)


,order_id,order_purchase_timestamp,order_estimated_delivery_date,y,distancia_km,mismo_estado,customer_state,seller_state,volumen_dm3,peso_g,...,n_vendedores,product_category_name,precio_total,flete_total,cuotas_max,ratio_flete_precio,dia_semana,hora,mes,es_fin_de_semana
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-18,8.436574,18.576110,1,SP,SP,1.976,500.0,...,1,utilidades_domesticas,29.99,8.72,1.0,0.290764,0,10,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-13,13.782037,851.495069,0,BA,SP,4.693,400.0,...,1,perfumaria,118.70,22.76,1.0,0.191744,1,20,7,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-09-04,9.394213,514.410666,0,GO,SP,9.576,420.0,...,1,automotivo,159.90,19.22,3.0,0.120200,2,8,8,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-15,13.208750,1822.226336,0,RN,MG,6.000,450.0,...,1,pet_shop,45.00,27.20,1.0,0.604444,5,19,11,1
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-26,2.873877,29.676625,1,SP,SP,11.475,250.0,...,1,papelaria,19.90,8.72,1.0,0.438191,1,21,2,0


## 5. Requisitos de Procesamiento de Datos y Limpieza

> Pendiente. Lo que pide la consigna:

### Tratamiento de Outliers
- [ ] Identificar y remover valores atípicos en `y` (valores negativos o inconsistentes, tiempos mayores a 90-120 días).
- [ ] Tratar valores atípicos en dimensiones físicas y fletes con métricas robustas (IQR o percentiles 1 % y 99 %).

### Ingeniería de Variables y Codificación
- [ ] One-Hot Encoding o Target Encoding para `customer_state`, `seller_state`, `product_category_name`.

### Escalado
- [ ] `log(1 + x)` en variables sesgadas, o `StandardScaler` / `RobustScaler`.

## 6. Estrategia de Validación del Modelo

> Pendiente. División temporal (sin `train_test_split` aleatorio):

- [ ] Entrenamiento: octubre de 2016 a mayo de 2018.
- [ ] Prueba: junio de 2018 a agosto de 2018.

## 7. Modelos a Implementar y Comparar

> Pendiente. Al menos tres algoritmos:

- [ ] Baseline: Regresión Lineal / Ridge / Lasso.
- [ ] Basado en árboles: Random Forest Regressor.
- [ ] Boosting: XGBoost / LightGBM / CatBoost.

## 8. Métricas de Evaluación Obligatorias

> Pendiente. Sobre el conjunto de prueba:

- [ ] MAE
- [ ] RMSE
- [ ] R²
- [ ] MAPE

## 9. Entregables Esperados

> Pendiente.

- [ ] Código limpio, documentado y reproducible (este notebook).
- [ ] Gráfico de análisis de residuos del mejor modelo.
- [ ] Importancia de características: top 10 factores.
- [ ] Conclusión de negocio: predicción del modelo vs. `order_estimated_delivery_date`.